# CampusCare AI — Big Data Student Support Message Triage

This notebook walks through the project workflow: data loading, EDA, preprocessing, baseline, deep-learning results, and error analysis.

## 1. Problem framing

The task is to classify anonymous student messages into `normal`, `academic_stress`, `emotional_distress`, or `urgent_human_review`. The system is a human-in-the-loop routing assistant, not a clinical diagnosis tool.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from data_utils import load_project_data, make_splits
from config import LABELS, OUTPUT_DIR

df = load_project_data()
df.shape, df.head()

## 2. Dataset distribution

In [ ]:
df["label"].value_counts().reindex(LABELS)

In [ ]:
df[["text_length", "word_count"]].describe()

## 3. Train/validation/test split

In [ ]:
train_df, val_df, test_df = make_splits(df)
print(len(train_df), len(val_df), len(test_df))
print(train_df["label"].value_counts(normalize=True).reindex(LABELS))

## 4. Run experiments

Run these cells from the project root. The included zip already has generated outputs, but you can retrain to reproduce them.

In [ ]:
# !python src/train_baseline.py
# !python src/train_deep.py --models TextCNN --epochs 2 --max-train-samples 12000
# !python src/train_deep.py --models BiLSTM --epochs 1 --max-train-samples 4000
# !python src/make_plots.py

## 5. Results

In [ ]:
baseline = pd.read_csv(ROOT / "outputs" / "baseline_results_summary.csv")
deep = pd.read_csv(ROOT / "outputs" / "deep_results_summary.csv")
pd.concat([baseline, deep], ignore_index=True)

## 6. Error analysis on challenge split

In [ ]:
mistakes_path = ROOT / "outputs" / "baseline_challenge_mistakes.csv"
if mistakes_path.exists():
    mistakes = pd.read_csv(mistakes_path)
    display(mistakes[["text", "label", "pred"]].head(20))
    print("Number of saved mistakes:", len(mistakes))

## 7. Interpretation notes

- The standard synthetic test split is easy, so very high scores are expected.
- The challenge split is harder and better for discussing real model behavior.
- The TF-IDF baseline beating undertrained deep models is a valid finding.
- For a stronger final version, train BiLSTM longer or fine-tune DistilBERT on a GPU.